# PageZero SRE Agent — GRPO Training

Trains a Qwen3-0.6B agent to diagnose and fix production incidents using GRPO.

**Stack:** TRL 1.2+ | transformers 5.x | bitsandbytes 4-bit | openenv-core

**Run all cells in order from top to bottom.**

## 1) Install Dependencies

In [1]:
import subprocess
import sys
from importlib import metadata

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)

# Core packages — no unsloth, no vllm (they conflict with TRL 1.x / transformers 5.x)
core_packages = [
    'trl>=0.29.0',
    'transformers>=5.2.0',
    'huggingface-hub>=1.5.0,<2.0',
    'datasets',
    'accelerate',
    'bitsandbytes',
    'matplotlib',
    'pandas',
    'peft',
    'jmespath',
    'nest_asyncio',
    'liger-kernel',
    'trackio',
    'openenv-core[core]>=0.2.1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *core_packages], check=True)

# Verify
print('-' * 40)
for pkg in ['transformers', 'trl', 'huggingface-hub', 'datasets', 'peft', 'bitsandbytes', 'liger-kernel', 'trackio']:
    try:
        print(f'{pkg}: {metadata.version(pkg)}')
    except Exception:
        print(f'{pkg}: NOT INSTALLED')

----------------------------------------
transformers: 5.6.2
trl: 1.2.0
huggingface-hub: 1.12.0
datasets: 4.8.4
peft: 0.19.1
bitsandbytes: 0.49.2
liger-kernel: 0.7.0
trackio: 0.25.0





### ⚠️ After running the cell above: Runtime → Restart runtime, then continue from Cell 2

## 2) Configuration

In [2]:
import os

IN_COLAB = 'COLAB_GPU' in os.environ

# --- HuggingFace token (optional unless pushing to hub) ---
try:
    from google.colab import userdata
    if 'HF_TOKEN' not in os.environ:
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
except Exception:
    pass

if 'HF_TOKEN' not in os.environ:
    print('WARNING: HF_TOKEN not found. Needed only for push_to_hub.')

# --- Backend mode selector ---
BACKEND_MODE = 'deployed_space'  # local_same_machine | local_via_tunnel | deployed_space | custom_url

LOCAL_SAME_MACHINE_URL = 'http://localhost:8000'
LOCAL_TUNNEL_URL = 'https://brave-pots-throw.loca.lt/'
DEPLOYED_SPACE_URL = 'https://neokazuha-pagezero-agent.hf.space'
CUSTOM_ENV_URL = os.environ.get('PAGEZERO_ENV_URL', '').strip()

if BACKEND_MODE == 'local_same_machine':
    ENV_URL = LOCAL_SAME_MACHINE_URL
elif BACKEND_MODE == 'local_via_tunnel':
    ENV_URL = LOCAL_TUNNEL_URL
elif BACKEND_MODE == 'deployed_space':
    ENV_URL = DEPLOYED_SPACE_URL
elif BACKEND_MODE == 'custom_url':
    if not CUSTOM_ENV_URL:
        raise ValueError("BACKEND_MODE='custom_url' but PAGEZERO_ENV_URL is empty.")
    ENV_URL = CUSTOM_ENV_URL
else:
    raise ValueError(f'Unknown BACKEND_MODE: {BACKEND_MODE}')

if IN_COLAB and BACKEND_MODE == 'local_same_machine':
    print('WARNING: localhost points to the Colab VM, not your laptop.')
    print('Use local_via_tunnel or deployed_space for Colab.')

# --- Repo / checkout config ---
REPO_URL = 'https://huggingface.co/spaces/pranayy/pagezero_agent'
USE_EXISTING_CHECKOUT = not IN_COLAB
EXISTING_CHECKOUT_DIR = os.getcwd() if not IN_COLAB else '/content/PageZero'
CLONE_DIR = '/content/PageZero' if IN_COLAB else EXISTING_CHECKOUT_DIR

# --- Model config ---
USE_UNSLOTH_MODEL = False  # True will try 4-bit Unsloth model id
UNSLOTH_MODEL_ID = 'unsloth/Qwen3-0.6B-bnb-4bit'
BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'
MODEL_ID = UNSLOTH_MODEL_ID if USE_UNSLOTH_MODEL else BASE_MODEL_ID
FALLBACK_MODEL_ID = BASE_MODEL_ID

# --- Training hyperparameters ---
NUM_EPISODES = 4 # Reduced for quicker testing
# GRPO group size (g): completions sampled per prompt that compete inside a group.
# Larger g → lower advantage variance and more stable gradients, at higher GPU cost.
# Bumped from 12 → 16 as the convergence/variance experiment.
NUM_GENERATIONS = 8
G_PARAM = NUM_GENERATIONS  # alias to make the "g" knob explicit in logs/plots
MAX_TURNS = 15
MAX_NEW_TOKENS = 256 # Reduced to mitigate OutOfMemoryError
VLLM_MODE = 'colocate'
OUTPUT_DIR_BASE = 'outputs/pagezero-colab'

# --- KL-divergence regularization (β) ---
# β controls how strongly GRPO pulls the trained policy toward the frozen
# reference policy. Lower β → more exploration but risks policy collapse /
# heavy drift; higher β → safer but slower learning. Tuned upward from the
# old 0.01 → 0.04 to dampen the high-variance SRE reward signal while still
# allowing the policy to discover new tool-use sequences.
KL_BETA = 0.04

# --- Curriculum learning stages ---
# Train sequentially on easy → medium → hard scenario pools. Each stage uses
# its own task_id pool from server.llm_designer (warmup→medium→hard). The
# trainer is reused across stages and a checkpoint is saved at every
# transition so we can rollback / compare per-difficulty performance.
CURRICULUM_STAGES = [
    {
        'name': 'easy',
        'task_ids': ['task_1', 'task_2', 'task_3', 'task_4', 'task_5'],
        'episodes': NUM_EPISODES,
        'max_difficulty': 0.4,
    },
    {
        'name': 'medium',
        'task_ids': ['task_6', 'task_7', 'task_8', 'task_9', 'task_10'],
        'episodes': NUM_EPISODES,
        'max_difficulty': 0.6,
    },
    {
        'name': 'hard',
        'task_ids': ['task_11', 'task_12'],
        'episodes': NUM_EPISODES,
        'max_difficulty': 1.0,
    },
]

# --- Trajectory logging ---
# Per-episode action trajectories (every (tool, args, reward) tuple) are
# written as JSONL for offline analysis: failure mode mining, credit-assignment
# inspection, and reward-function audits.
TRAJECTORY_LOG_NAME = 'trajectories.jsonl'

# --- Reward audit guards ---
# Bounds we expect the trajectory-level total reward to live in; anything
# outside flags a reward-shaping bug or a numerical issue. Sized for the
# corrected llm_judge reward scheme: per-step in [-0.30, +0.30] over up to
# MAX_TURNS steps + an uncapped terminal score in roughly [-1.0, +1.0]
# → worst-case episode total ≈ ±(0.30 * MAX_TURNS + 1.0). We add ~30%
# headroom so genuine extremes do not get clipped (clipping silently
# distorts the GRPO advantage and was the previous bug).
REWARD_AUDIT_MIN = -6.0
REWARD_AUDIT_MAX = 6.0

# --- Hub push config ---
PUSH_TO_HUB = False
HUB_REPO = 'pranayy/pagezero_agent'

print(f'IN_COLAB        : {IN_COLAB}')
print(f'BACKEND_MODE    : {BACKEND_MODE}')
print(f'ENV_URL         : {ENV_URL}')
print(f'MODEL_ID        : {MODEL_ID}')
print(f'EPISODES/STAGE  : {NUM_EPISODES}')
print(f'GENERATIONS (g) : {NUM_GENERATIONS}')
print(f'KL β            : {KL_BETA}')
print(f'CURRICULUM      : {[s["name"] for s in CURRICULUM_STAGES]}')

IN_COLAB        : True
BACKEND_MODE    : deployed_space
ENV_URL         : https://neokazuha-pagezero-agent.hf.space
MODEL_ID        : Qwen/Qwen3-0.6B
EPISODES/STAGE  : 4
GENERATIONS (g) : 8
KL β            : 0.04
CURRICULUM      : ['easy', 'medium', 'hard']


## 3) Clone Repo & Install

In [3]:
import os
import subprocess

if IN_COLAB:
    if not os.path.exists(CLONE_DIR):
        subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    os.chdir(CLONE_DIR)
    subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)

print(f'Working directory: {os.getcwd()}')

Working directory: /content/PageZero


## 4) Test Backend Connection

In [4]:
import requests
import asyncio
import sys, os

if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from PageZero.client import PageZeroEnvClient
from PageZero.models import PageZeroAction

print(f'Testing backend: {ENV_URL}')
url = ENV_URL.rstrip('/') + '/health'
try:
    r = requests.get(url, timeout=10)
    print(f'{url} -> {r.status_code}')
except Exception as e:
    print(f'{url} -> FAILED: {e}')

async def run_env_interaction():
    env = PageZeroEnvClient(base_url=ENV_URL, message_timeout_s=300.0)
    try:
        reset_result = await env.reset()
        obs = reset_result.observation
        print('Reset successful')
        print(f'Step {obs.step}/{obs.max_steps}')
        print(f'Active alerts: {obs.active_alerts}')

        step_result = await env.step(PageZeroAction(tool='check_alerts', args={}))
        print(f'Step reward: {step_result.reward}')
        print(f'Done: {step_result.done}')
    finally:
        await env.close()

await run_env_interaction()

Testing backend: https://neokazuha-pagezero-agent.hf.space
https://neokazuha-pagezero-agent.hf.space/health -> 200
Reset successful
Step 0/15
Active alerts: ['CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 95% — investigate active queries in PostgreSQL']
Step reward: 0.1
Done: False


## 5) Import Training Utilities

In [5]:
import csv
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from PageZero.train import SYSTEM_PROMPT, plot_rewards

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

print('Imports OK.')
print('System prompt preview:')
print(SYSTEM_PROMPT[:220] + '...')

Imports OK.
System prompt preview:
You are a Staff SRE on-call. Diagnose and fix the cascading incident across the Application, PostgreSQL, and Redis cache.

To use a tool, you MUST use this exact format:
<tool_call>
{"name": "check_alerts", "arguments": ...


## 6) Define PageZeroToolEnv (async-safe for Colab)

In [12]:
import asyncio, threading, time, random, os
from PageZero.client import PageZeroEnvClient
from PageZero.models import PageZeroAction

# ── Background event loop (avoids Jupyter's "loop already running" error) ──
_bg_loop = asyncio.new_event_loop()
threading.Thread(target=_bg_loop.run_forever, daemon=True).start()

def _sync(coro, timeout=360):
    """Submit async coroutine to background loop, block until done."""
    return asyncio.run_coroutine_threadsafe(coro, _bg_loop).result(timeout=timeout)


# Pool of (tool_prefix → bucket) used by the audit/track logic to classify
# whether a step is a diagnosis step or a fix step. Keep in sync with
# server/llm_judge.py _FIX_TOOLS.
_FIX_TOOL_PREFIXES = (
    'pg_cancel', 'pg_create', 'pg_vacuum',
    'docker_restart', 'rollback_deploy', 'redis_flush_db',
)


class PageZeroToolEnv:
    """
    Colab-safe environment wrapper for GRPOTrainer(environment_factory=...).
    Uses a dedicated background asyncio thread to bridge the async openenv
    WebSocket client into synchronous calls that TRL expects.
    Creates a fresh WebSocket per episode to avoid connection timeouts.

    Per-episode this wrapper also captures a structured *trajectory*:
        [{step, tool, args, reward, sla_status, done, classification}, ...]
    so the trainer cell can dump it as JSONL for trajectory-level reward
    auditing, credit-assignment debugging and failure-pattern mining.
    """

    BASE_URL: str = ENV_URL
    # Curriculum hooks — set by the training loop before each stage.
    STAGE: str = 'default'
    STAGE_TASK_IDS: list = []
    _instances: list = []

    # Set to truthy (env var or class attr) to print a one-line trace for
    # every tool call. Useful when debugging "all rewards are 0" runs.
    VERBOSE_REWARDS: bool = bool(int(os.environ.get('PZ_VERBOSE_REWARDS', '1')))

    def __init__(self):
        self._client = None
        self.total_reward = 0.0
        # NOTE: these are *accumulators* — every per-step reward in the
        # category is added (the previous bug overwrote them, so only the
        # last step in the bucket counted and the terminal bonus leaked
        # into the wrong bucket).
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0   # captured separately on done
        self.is_done = False
        self._episode_logged = False  # dedup guard for reward logging
        # Trajectory-level state (full action sequence for the episode).
        self.trajectory: list = []
        self.step_rewards: list = []
        self.task_id: str = ''
        self.scenario_name: str = ''
        self.final_score = None       # canonical (validator) terminal score
        self.last_sla_status: str = 'OK'
        self.is_resolved: bool = False
        self.start_ts = time.time()
        self.end_ts = None
        self.__class__._instances.append(self)

    @classmethod
    def _close_all(cls):
        for env in cls._instances:
            if env._client:
                try: _sync(env._client.close(), timeout=10)
                except: pass
        cls._instances.clear()

    @classmethod
    def set_stage(cls, name: str, task_ids: list):
        """Bind a curriculum stage to all future episode resets.

        Args:
            name: The name of the stage.
            task_ids: A list of task IDs for the stage.
        """
        cls.STAGE = name
        cls.STAGE_TASK_IDS = list(task_ids)

    # ---- Lifecycle ----

    def reset(self, **kwargs) -> str | None:
        """Reset environment state for a new episode.

        If the curriculum stage has a task pool bound, one task_id is sampled
        and forwarded to the server so this episode is constrained to the
        chosen difficulty bucket.

        Returns:
            Initial observation text appended to the user prompt.
        """
        if self._client:
            try: _sync(self._client.close(), timeout=10)
            except: pass
            self._client = None
            time.sleep(0.3)

        self._client = PageZeroEnvClient(
            base_url=self.BASE_URL, message_timeout_s=300.0)

        reset_kwargs = {}
        chosen_task_id = ''
        if self.STAGE_TASK_IDS:
            chosen_task_id = random.choice(self.STAGE_TASK_IDS)
            reset_kwargs['task_id'] = chosen_task_id

        # Some server builds may not accept the task_id kwarg yet; fall back
        # to a plain reset() so older deployments keep working.
        try:
            result = _sync(self._client.reset(**reset_kwargs))
        except Exception:
            result = _sync(self._client.reset())
            chosen_task_id = ''

        self.total_reward = 0.0
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0
        self.is_done = bool(result.done)
        self._episode_logged = False
        self.trajectory = []
        self.step_rewards = []
        self.task_id = chosen_task_id
        self.scenario_name = getattr(result.observation, 'tool_output', '')[:120]
        self.final_score = None
        self.last_sla_status = getattr(result.observation, 'sla_status', 'OK') or 'OK'
        self.is_resolved = False
        self.start_ts = time.time()
        self.end_ts = None
        if self.VERBOSE_REWARDS:
            print(f'[env-reset] stage={self.STAGE} task={chosen_task_id or "auto"} '
                  f'scenario_preview={self.scenario_name[:60]!r}')
        return self._format_observation(result.observation, reward=0.0)

    def _format_observation(self, obs, reward):
        tool_output = getattr(obs, 'tool_output', '') or ''
        alerts = getattr(obs, 'active_alerts', []) or []
        alert_text = '\n'.join(alerts) if alerts else 'None'
        sla = getattr(obs, 'sla_status', 'OK')
        rev = getattr(obs, 'revenue_loss_usd', 0.0)
        dt = getattr(obs, 'downtime_minutes', 0.0)
        step = getattr(obs, 'step', 0)
        mx = getattr(obs, 'max_steps', 15)
        hint = getattr(obs, 'hint', '') or ''
        text = (
            f'{tool_output}\n\n'
            f'CURRENT ALERTS:\n{alert_text}\n\n'
            f'SLA STATUS: {sla}\n'
            f'REVENUE LOST: ${rev}\n'
            f'DOWNTIME: {dt} minutes\n'
            f'STEP REWARD: {reward:.4f}\n'
            f'STEP: {step}/{mx}'
        )
        if hint:
            text += f'\nHINT: {hint}'
        return text

    def _run_tool(self, tool, args):
        if self.is_done and tool != 'done':
            raise ValueError('Episode already done.')

        # Retry once on connection drop
        for attempt in range(2):
            try:
                result = _sync(
                    self._client.step(PageZeroAction(tool=tool, args=args)))
                break
            except Exception as e:
                if attempt == 0 and 'Closed' in type(e).__name__:
                    print(f'  [reconnect] WebSocket dropped, reconnecting...')
                    self._client = PageZeroEnvClient(
                        base_url=self.BASE_URL, message_timeout_s=300.0)
                    _sync(self._client.reset())
                    continue
                raise

        # The server's `obs.reward` already contains: per-step phase reward
        # (+ ERROR penalty, if any) + terminal training bonus on the done
        # step. We split that into a per-step component and a terminal
        # component so the buckets stay attributable.
        obs = result.observation
        reward = float(result.reward or 0.0)
        was_done_step = bool(result.done)
        canonical_final = getattr(obs, 'final_score', None)
        try:
            canonical_final = float(canonical_final) if canonical_final is not None else None
        except Exception:
            canonical_final = None

        # Estimate the terminal training bonus the server added on `done`. It
        # is the (uncapped) raw judge score. We can't read it directly, so we
        # treat any non-zero gap between the per-step component and `reward`
        # as the terminal contribution. Server-side that decomposition is
        # exact; on the client we only need an approximate split for logging.
        per_step_component = reward
        terminal_component = 0.0
        if was_done_step:
            # Heuristic: if the env returned a canonical score, the terminal
            # component is roughly (raw_training ≈ 2*canonical - 1). We use
            # this only for *bucket attribution*; the optimizer always uses
            # `total_reward = sum(reward)` which is exact.
            if canonical_final is not None:
                approx_terminal = round(2.0 * canonical_final - 1.0, 3)
                # Cap the inferred terminal so a noisy estimate can't dominate.
                approx_terminal = max(-1.0, min(1.0, approx_terminal))
                terminal_component = approx_terminal
                per_step_component = reward - approx_terminal
            self.terminal_reward = terminal_component

        self.total_reward += reward
        self.is_done = was_done_step

        is_fix = tool.startswith(_FIX_TOOL_PREFIXES)
        classification = 'fix' if is_fix else 'diagnose'
        # ACCUMULATE (don't overwrite) — previous code only kept the last
        # step in each bucket so multi-step fixes / repeated investigations
        # were lost from the reward signal.
        if is_fix:
            self.fix_reward += per_step_component
        else:
            self.diagnosis_reward += per_step_component

        sla_status = getattr(obs, 'sla_status', 'OK') or 'OK'
        self.last_sla_status = sla_status
        raw_out = (getattr(obs, 'tool_output', '') or '')[:400]
        self.step_rewards.append(reward)
        self.trajectory.append({
            'step': len(self.trajectory) + 1,
            'tool': tool,
            'args': args,
            'reward': reward,
            'per_step_reward': per_step_component,
            'terminal_reward_component': terminal_component,
            'cumulative_reward': self.total_reward,
            'sla_status': sla_status,
            'revenue_loss_usd': float(getattr(obs, 'revenue_loss_usd', 0.0) or 0.0),
            'downtime_minutes': float(getattr(obs, 'downtime_minutes', 0.0) or 0.0),
            'classification': classification,
            'is_done': self.is_done,
            'output_snippet': raw_out,
        })

        if self.is_done:
            self.end_ts = time.time()
            self.final_score = canonical_final
            self.is_resolved = bool(getattr(obs, 'is_done', False))

        if self.VERBOSE_REWARDS:
            tag = 'FIX' if is_fix else 'DIAG'
            terminal_note = f' term={terminal_component:+.3f}' if was_done_step else ''
            print(
                f'  [step {len(self.trajectory):>2}] {tag:<4s} {tool:<22s} '
                f'r={reward:+.3f} cum={self.total_reward:+.3f}{terminal_note}'
            )

        return self._format_observation(obs, reward=reward)

    def trajectory_payload(self) -> dict:
        """Return a single dict representing the *full* episode trajectory.

        This is what gets serialized to the trajectories.jsonl log so we can
        audit credit assignment and reward shaping after training.
        """
        n_steps = len(self.trajectory)
        sum_step_rewards = float(sum(self.step_rewards))
        # The done step carries (per_step + terminal). We logged the terminal
        # split per-step, so the bucket sum should equal total_reward.
        bucket_sum = float(self.diagnosis_reward + self.fix_reward + self.terminal_reward)
        return {
            'stage': self.STAGE,
            'task_id': self.task_id,
            'scenario_name': self.scenario_name,
            'num_steps': n_steps,
            'is_resolved': bool(self.is_resolved),
            'final_score': self.final_score,           # canonical (validator)
            'total_reward': float(self.total_reward),  # training signal
            'sum_step_rewards': sum_step_rewards,
            'terminal_reward': float(self.terminal_reward),
            # Kept for back-compat with prior runs / log readers.
            'terminal_reward_delta': float(self.terminal_reward),
            'diagnosis_reward': float(self.diagnosis_reward),
            'fix_reward': float(self.fix_reward),
            'bucket_sum': bucket_sum,
            'bucket_total_mismatch': round(self.total_reward - bucket_sum, 4),
            'last_sla_status': self.last_sla_status,
            'wallclock_s': (self.end_ts or time.time()) - self.start_ts,
            'tool_sequence': [t['tool'] for t in self.trajectory],
            'trajectory': self.trajectory,
        }

    # ═══ Tools (public methods → TRL auto-discovers these) ═══

    def check_alerts(self) -> str:
        """Check active incident alerts.

        Returns:
            Current alert information.
        """
        return self._run_tool('check_alerts', {})

    def get_service_metrics(self, service: str = 'app') -> str:
        """Get service metrics.

        Args:
            service: Service name (app, redis, postgres).

        Returns:
            Service metrics.
        """
        return self._run_tool('get_service_metrics', {'service': service})

    def get_error_rate(self) -> str:
        """Get application error rate.

        Returns:
            Error rate summary.
        """
        return self._run_tool('get_error_rate', {})

    def read_app_logs(self, lines: int = 200) -> str:
        """Read recent application logs.

        Args:
            lines: Number of log lines to read.

        Returns:
            Log output.
        """
        return self._run_tool('read_app_logs', {'lines': lines})

    def search_logs(self, pattern: str) -> str:
        """Search logs for a pattern.

        Args:
            pattern: Search text.

        Returns:
            Matching log lines.
        """
        return self._run_tool('search_logs', {'pattern': pattern})

    def get_recent_deploys(self) -> str:
        """List recent deployments.

        Returns:
            Deployment history.
        """
        return self._run_tool('get_recent_deploys', {})

    def rollback_deploy(self) -> str:
        """Rollback the latest deployment.

        Returns:
            Rollback status.
        """
        return self._run_tool('rollback_deploy', {})

    def curl_endpoint(self, url: str) -> str:
        """Curl an HTTP endpoint.

        Args:
            url: Endpoint URL to call.

        Returns:
            HTTP response.
        """
        return self._run_tool('curl_endpoint', {'url': url})

    def pg_stat_activity(self) -> str:
        """Inspect PostgreSQL active sessions.

        Returns:
            Active session information.
        """
        return self._run_tool('pg_stat_activity', {})

    def pg_locks(self) -> str:
        """Inspect PostgreSQL locks.

        Returns:
            Lock diagnostics.
        """
        return self._run_tool('pg_locks', {})

    def pg_explain_analyze(self, query: str) -> str:
        """Run EXPLAIN ANALYZE on a SQL query.

        Args:
            query: The SQL query to analyze.

        Returns:
            Query execution plan.
        """
        return self._run_tool('pg_explain_analyze', {'query': query})

    def pg_stat_statements(self) -> str:
        """Inspect pg_stat_statements.

        Returns:
            Statement statistics.
        """
        return self._run_tool('pg_stat_statements', {})

    def pg_cancel_query(self, pid: int) -> str:
        """Cancel a running PostgreSQL query.

        Args:
            pid: Process ID of the query to cancel.

        Returns:
            Cancellation result.
        """
        return self._run_tool('pg_cancel_query', {'pid': pid})

    def pg_create_index(self, table: str, column: str) -> str:
        """Create an index on a PostgreSQL table.

        Args:
            table: Table name.
            column: Column name.

        Returns:
            Index creation result.
        """
        return self._run_tool('pg_create_index', {'table': table, 'column': column})

    def pg_vacuum(self, table: str) -> str:
        """Run VACUUM on a PostgreSQL table.

        Args:
            table: Table name.

        Returns:
            Vacuum status.
        """
        return self._run_tool('pg_vacuum', {'table': table})

    def pg_show_tables(self) -> str:
        """List PostgreSQL tables.

        Returns:
            Table list.
        """
        return self._run_tool('pg_show_tables', {})

    def redis_info(self) -> str:
        """Get Redis server INFO.

        Returns:
            Redis diagnostics.
        """
        return self._run_tool('redis_info', {})

    def redis_slowlog(self) -> str:
        """Inspect Redis slowlog.

        Returns:
            Slowlog entries.
        """
        return self._run_tool('redis_slowlog', {})

    def redis_keys(self, pattern: str = '*') -> str:
        """List Redis keys matching a pattern.

        Args:
            pattern: Key glob pattern.

        Returns:
            Matching keys.
        """
        return self._run_tool('redis_keys', {'pattern': pattern})

    def redis_flush_db(self) -> str:
        """Flush the Redis database.

        Returns:
            Flush result.
        """
        return self._run_tool('redis_flush_db', {})

    def redis_get_key(self, key: str) -> str:
        """Get the value of a Redis key.

        Args:
            key: Redis key name.

        Returns:
            Key value.
        """
        return self._run_tool('redis_get_key', {'key': key})

    def docker_ps(self) -> str:
        """List running Docker containers.

        Returns:
            Container list.
        """
        return self._run_tool('docker_ps', {})

    def docker_stats(self, container: str) -> str:
        """Get Docker container resource stats.

        Args:
            container: Container name.

        Returns:
            Resource statistics.
        """
        return self._run_tool('docker_stats', {'container': container})

    def docker_restart(self, container: str) -> str:
        """Restart a Docker container.

        Args:
            container: Container name.

        Returns:
            Restart result.
        """
        return self._run_tool('docker_restart', {'container': container})

    def docker_logs(self, container: str) -> str:
        """Read Docker container logs.

        Args:
            container: Container name.

        Returns:
            Container logs.
        """
        return self._run_tool('docker_logs', {'container': container})

    def check_disk_usage(self) -> str:
        """Check disk usage on the host.

        Returns:
            Disk usage summary.
        """
        return self._run_tool('check_disk_usage', {})

    def diagnose_root_cause(self, root_cause: str) -> str:
        """Record the diagnosed root cause of the incident.

        Args:
            root_cause: Description of the root cause.

        Returns:
            Acknowledgement.
        """
        return self._run_tool('diagnose_root_cause', {'root_cause': root_cause})

    def write_postmortem(self, root_cause: str, impact: str, fix_applied: str, prevention: str) -> str:
        """Write an incident postmortem report.

        Args:
            root_cause: The underlying cause of the incident.
            impact: Scope of the disruption.
            fix_applied: Steps taken to resolve.
            prevention: How to prevent recurrence.

        Returns:
            Acknowledgement.
        """
        return self._run_tool('write_postmortem', {
            'root_cause': root_cause, 'impact': impact,
            'fix_applied': fix_applied, 'prevention': prevention})

    def done(self) -> str:
        """Mark the incident as resolved and complete the episode.

        Returns:
            Final message.
        """
        return self._run_tool('done', {})


# Quick smoke test
print('Smoke-testing PageZeroToolEnv...')
_test_env = PageZeroToolEnv()
_test_obs = _test_env.reset()
print(f'Reset OK. Observation preview: {_test_obs[:100]}...')
_sync(_test_env._client.close(), timeout=10)
PageZeroToolEnv._instances.clear()
print('Smoke test PASSED ✓')


Smoke-testing PageZeroToolEnv...
[env-reset] stage=default task=auto scenario_preview='🚨 ALERT: CRITICAL: Redis memory usage > 95%, OOM errors in a'
Reset OK. Observation preview: 🚨 ALERT: CRITICAL: Redis memory usage > 95%, OOM errors in app logs — check redis_info and flush orp...
Smoke test PASSED ✓


## 7) Build GRPO Trainer

In [13]:
import torch, gc, csv, json, math
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig
from datasets import Dataset
from transformers import AutoTokenizer
from pathlib import Path
from datetime import datetime

# --- 1. PREVENT OOM on re-runs ---
try:
    if 'trainer' in locals():
        del trainer
    if 'model' in locals():
        del model
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

# --- 2. Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 3. Dataset (one prompt per episode in the current curriculum stage) ---
prompt_messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'Diagnose and fix this production incident.'},
]

def make_stage_dataset(num_episodes: int) -> Dataset:
    """Build a fresh dataset for one curriculum stage."""
    return Dataset.from_dict(
        {'prompt': [prompt_messages for _ in range(num_episodes)]}
    )

dataset = make_stage_dataset(NUM_EPISODES)

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
output_dir = Path(f'{OUTPUT_DIR_BASE}-{timestamp}')
output_dir.mkdir(parents=True, exist_ok=True)

# --- 4. Reward + trajectory logging plumbing -----------------------------
# We log two parallel artifacts per training run:
#   • reward_log.csv         → one row per episode with totals (analytics)
#   • trajectories.jsonl     → one line per episode with the *full* action
#                              trajectory (for credit assignment + reward audit)
# Both can be redirected to a per-stage subfolder by `bind_logs(...)` so the
# curriculum loop in the next cell can keep stages cleanly separated.

_LOG_STATE = {
    'reward_csv': output_dir / 'reward_log.csv',
    'trajectory_jsonl': output_dir / TRAJECTORY_LOG_NAME,
    'episode_counter': 0,
    'audit_counter': 0,
    'stage': 'init',
}

REWARD_HEADER = [
    'episode', 'stage', 'task_id', 'total_reward',
    'diagnosis_reward', 'fix_reward', 'final_score',
    'num_steps', 'is_resolved', 'sum_step_rewards',
    'terminal_reward_delta', 'last_sla_status', 'timestamp',
]

def _ensure_csv_header(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists() or path.stat().st_size == 0:
        with open(path, 'w', newline='') as f:
            csv.writer(f).writerow(REWARD_HEADER)

def bind_logs(reward_csv: Path, trajectory_jsonl: Path, stage: str):
    """Point the reward + trajectory logs at a new (per-stage) location."""
    _LOG_STATE['reward_csv'] = Path(reward_csv)
    _LOG_STATE['trajectory_jsonl'] = Path(trajectory_jsonl)
    _LOG_STATE['stage'] = stage
    _ensure_csv_header(_LOG_STATE['reward_csv'])
    _LOG_STATE['trajectory_jsonl'].parent.mkdir(parents=True, exist_ok=True)
    _LOG_STATE['trajectory_jsonl'].touch(exist_ok=True)

bind_logs(_LOG_STATE['reward_csv'], _LOG_STATE['trajectory_jsonl'], 'init')

def _log_trajectory(env):
    """Write per-episode CSV row + per-episode trajectory JSONL line."""
    payload = env.trajectory_payload()
    payload['stage'] = _LOG_STATE['stage'] or payload.get('stage', '')
    _LOG_STATE['episode_counter'] += 1
    ep_idx = _LOG_STATE['episode_counter']
    payload['episode'] = ep_idx
    payload['timestamp'] = datetime.now().isoformat()

    with open(_LOG_STATE['reward_csv'], 'a', newline='') as f:
        csv.writer(f).writerow([
            ep_idx,
            payload['stage'],
            payload.get('task_id', ''),
            payload['total_reward'],
            payload['diagnosis_reward'],
            payload['fix_reward'],
            payload.get('final_score', ''),
            payload['num_steps'],
            int(bool(payload['is_resolved'])),
            payload['sum_step_rewards'],
            payload['terminal_reward_delta'],
            payload['last_sla_status'],
            payload['timestamp'],
        ])

    with open(_LOG_STATE['trajectory_jsonl'], 'a') as f:
        f.write(json.dumps(payload, default=str) + '\n')

    return payload

# --- 5. Reward audit -----------------------------------------------------
# Validates the reward signal we hand to GRPO before each optimizer step:
#   1. NaN / Inf → coerce to 0 and warn (would otherwise nuke the policy)
#   2. Out-of-range → clip into [REWARD_AUDIT_MIN, REWARD_AUDIT_MAX]
#   3. Per-group statistics (mean / std / min / max) printed every K calls.
#   4. Sanity check that |fix_reward| + |diagnosis_reward| ≈ |total_reward|.

def _reward_audit(values, label):
    """Sanitize, clip, and emit periodic stats for a reward batch."""
    cleaned = []
    for v in values:
        try:
            v = float(v)
        except Exception:
            v = 0.0
        if math.isnan(v) or math.isinf(v):
            print(f'  [reward-audit:{label}] WARN non-finite reward → 0.0')
            v = 0.0
        if v < REWARD_AUDIT_MIN or v > REWARD_AUDIT_MAX:
            print(f'  [reward-audit:{label}] WARN clip {v:+.3f} → '
                  f'[{REWARD_AUDIT_MIN}, {REWARD_AUDIT_MAX}]')
            v = max(REWARD_AUDIT_MIN, min(REWARD_AUDIT_MAX, v))
        cleaned.append(v)

    _LOG_STATE['audit_counter'] += 1
    # Print group statistics every call (cheap and useful during GRPO).
    if cleaned:
        mu = sum(cleaned) / len(cleaned)
        var = sum((x - mu) ** 2 for x in cleaned) / max(1, len(cleaned) - 1)
        sd = var ** 0.5
        print(
            f'  [reward-audit:{label}] n={len(cleaned)} '
            f'mean={mu:+.3f} std={sd:.3f} '
            f'min={min(cleaned):+.3f} max={max(cleaned):+.3f}'
        )
    return cleaned

# --- 6. Reward functions (audited + dedup-safe + trajectory-aware) -------
def reward_total(completions=None, environments=None, **kwargs):
    if not environments:
        return [0.0 for _ in range(len(completions) if completions else 0)]
    values = []
    for env in environments:
        total = float(getattr(env, 'total_reward', 0.0))
        # Log + dump trajectory exactly once per env episode.
        if not getattr(env, '_episode_logged', False):
            try:
                payload = _log_trajectory(env)
                # Cross-check trajectory-level total against per-step sum.
                terminal_delta = payload.get('terminal_reward_delta', 0.0)
                if abs(terminal_delta) > REWARD_AUDIT_MAX:
                    print(f"  [reward-audit:total] WARN terminal Δ={terminal_delta:+.3f}"
                          f" exceeds bound — review server-side terminal scoring.")
            except Exception as e:
                print(f'  [reward-audit:total] failed to log trajectory: {e}')
            env._episode_logged = True
        values.append(total)
    return _reward_audit(values, 'total')

def reward_diagnosis(completions=None, environments=None, **kwargs):
    if not environments:
        return [0.0 for _ in range(len(completions) if completions else 0)]
    values = [float(getattr(env, 'diagnosis_reward', 0.0)) for env in environments]
    return _reward_audit(values, 'diag')

def reward_fix(completions=None, environments=None, **kwargs):
    if not environments:
        return [0.0 for _ in range(len(completions) if completions else 0)]
    values = [float(getattr(env, 'fix_reward', 0.0)) for env in environments]
    return _reward_audit(values, 'fix')

# --- 7. Load model with 4-bit quantization ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

# --- 8. GRPO Config (Liger + Trackio enabled with compatibility guards) --
grpo_kwargs = dict(
    use_vllm=False,
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    lr_scheduler_type='cosine',
    max_tool_calling_iterations=MAX_TURNS,
    warmup_steps=2,
    max_grad_norm=1.0,
    gradient_accumulation_steps=1,  # Lowered to 1 to prevent OOM on T4 GPU
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,    # g — bumped to 16 (see Cell 2 notes)
    max_completion_length=512,
    logging_steps=1,
    save_strategy='steps',
    save_steps=10,
    temperature=1.0,
    report_to='trackio',
    run_name=f'pagezero-grpo-{timestamp}',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    save_total_limit=3,
    loss_type='dapo',
    mask_truncated_completions=True,
    beta=KL_BETA,                       # KL-divergence coefficient (β)
)

# Add optional args only if this TRL version supports them.
grpo_fields = set(getattr(GRPOConfig, '__dataclass_fields__', {}).keys())
if 'use_liger_kernel' in grpo_fields:
    grpo_kwargs['use_liger_kernel'] = True
if 'project' in grpo_fields:
    grpo_kwargs['project'] = 'pagezero-rl'

grpo_config = GRPOConfig(**grpo_kwargs)

# --- 9. LoRA Config ---
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

# --- 10. Initialize Trainer ---
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_diagnosis, reward_fix],
    train_dataset=dataset,
    args=grpo_config,
    peft_config=peft_config,
    environment_factory=PageZeroToolEnv,
)

print(f'Trainer initialized. Output path: {output_dir}')
print(f'  KL β             : {KL_BETA}')
print(f'  num_generations  : {NUM_GENERATIONS}  (g)')
print(f'  curriculum stages: {[s["name"] for s in CURRICULUM_STAGES]}')
print(f'  reward log       : {_LOG_STATE["reward_csv"]}')
print(f'  trajectory log   : {_LOG_STATE["trajectory_jsonl"]}')


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Trainer initialized. Output path: outputs/pagezero-colab-2026-04-25_19-07-10
  KL β             : 0.04
  num_generations  : 8  (g)
  curriculum stages: ['easy', 'medium', 'hard']
  reward log       : outputs/pagezero-colab-2026-04-25_19-07-10/reward_log.csv
  trajectory log   : outputs/pagezero-colab-2026-04-25_19-07-10/trajectories.jsonl


/tmp/ipykernel_1779/2216696972.py:247: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(


## 8) Train, Plot, and Save

In [8]:
PUSH_TO_HUB = True

In [14]:
import json, gc
import pandas as pd
from IPython.display import display, Image

print('Starting GRPO curriculum training...')
print(f'  Model        : {MODEL_ID}')
print(f'  KL β         : {KL_BETA}')
print(f'  g (num_gen)  : {NUM_GENERATIONS}')
print(f'  Episodes/stg : {NUM_EPISODES}')
print(f'  Environment  : {ENV_URL}')
print(f'  Stages       : {[s["name"] for s in CURRICULUM_STAGES]}')
print()


REWARD_COLUMNS = REWARD_HEADER  # defined in the trainer cell


def _normalize_reward_log(csv_path: Path) -> pd.DataFrame:
    """Return a tidy DataFrame for one stage's reward_log.csv."""
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return pd.DataFrame(columns=REWARD_COLUMNS)
    df = pd.read_csv(csv_path)
    for col in ('episode', 'total_reward', 'diagnosis_reward', 'fix_reward',
                'final_score', 'num_steps', 'is_resolved',
                'sum_step_rewards', 'terminal_reward_delta'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def _summarize_trajectories(jsonl_path: Path) -> dict:
    """Aggregate trajectory-level stats for a single curriculum stage."""
    if not jsonl_path.exists():
        return {}
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
    if not rows:
        return {}
    totals = [float(r.get('total_reward', 0.0)) for r in rows]
    resolved = [bool(r.get('is_resolved')) for r in rows]
    n_steps = [int(r.get('num_steps', 0)) for r in rows]
    tool_freq = {}
    for r in rows:
        for t in r.get('tool_sequence', []):
            tool_freq[t] = tool_freq.get(t, 0) + 1
    top_tools = sorted(tool_freq.items(), key=lambda kv: -kv[1])[:5]
    return {
        'episodes': len(rows),
        'mean_reward': sum(totals) / len(totals),
        'best_reward': max(totals),
        'worst_reward': min(totals),
        'resolution_rate': sum(resolved) / len(resolved),
        'mean_steps': sum(n_steps) / len(n_steps),
        'top_tools': top_tools,
    }


# --- Curriculum loop -----------------------------------------------------
STAGE_CHECKPOINTS: dict[str, Path] = {}
STAGE_SUMMARIES: dict[str, dict] = {}

try:
    for stage in CURRICULUM_STAGES:
        stage_name = stage['name']
        stage_dir = output_dir / f'stage_{stage_name}'
        stage_dir.mkdir(parents=True, exist_ok=True)

        # 1) Bind PageZeroToolEnv to this stage's task pool so every
        #    environment_factory() call samples a task_id from the right bucket.
        PageZeroToolEnv.set_stage(stage_name, stage['task_ids'])

        # 2) Redirect reward + trajectory logs into this stage's folder.
        bind_logs(
            reward_csv=stage_dir / 'reward_log.csv',
            trajectory_jsonl=stage_dir / TRAJECTORY_LOG_NAME,
            stage=stage_name,
        )

        # 3) Refresh dataset for this stage and reset trainer epoch state so
        #    repeated trainer.train() calls behave like fresh stages.
        trainer.train_dataset = make_stage_dataset(stage['episodes'])
        trainer.args.output_dir = str(stage_dir)
        trainer.args.run_name = f'pagezero-grpo-{timestamp}-{stage_name}'
        trainer.state.epoch = 0
        if hasattr(trainer.state, 'global_step'):
            trainer.state.global_step = 0
        if hasattr(trainer, '_episodes_completed'):
            trainer._episodes_completed = 0

        print('=' * 70)
        print(f'[curriculum] STAGE = {stage_name.upper()}  '
              f'(episodes={stage["episodes"]}, β={KL_BETA}, g={NUM_GENERATIONS}, '
              f'tasks={stage["task_ids"]})')
        print('=' * 70)

        try:
            trainer.train()
        finally:
            PageZeroToolEnv._close_all()

        # 4) Save a transition checkpoint for rollback / per-stage comparison.
        ckpt_dir = stage_dir / 'checkpoint'
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(ckpt_dir))
        STAGE_CHECKPOINTS[stage_name] = ckpt_dir
        print(f'[curriculum] {stage_name} → checkpoint saved at {ckpt_dir}')

        # 5) Per-stage reward plot + summary stats.
        stage_csv = stage_dir / 'reward_log.csv'
        try:
            plot_rewards(stage_csv, stage_dir / 'reward_plot.png')
        except Exception as e:
            print(f'WARNING: could not plot rewards for {stage_name}: {e}')

        summary = _summarize_trajectories(stage_dir / TRAJECTORY_LOG_NAME)
        STAGE_SUMMARIES[stage_name] = summary
        if summary:
            print(
                f'[curriculum] {stage_name} summary: '
                f"episodes={summary['episodes']} "
                f"mean={summary['mean_reward']:+.3f} "
                f"best={summary['best_reward']:+.3f} "
                f"resolved={summary['resolution_rate']:.0%} "
                f"avg_steps={summary['mean_steps']:.1f}"
            )

        # 6) Free GPU memory between stages.
        gc.collect()
        torch.cuda.empty_cache()

finally:
    PageZeroToolEnv._close_all()


# --- Final post-curriculum save + reporting -----------------------------
final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
print(f'\nFinal model saved to {final_dir}')

print('\n=== Curriculum stage checkpoints ===')
for name, path in STAGE_CHECKPOINTS.items():
    print(f'  {name:6s} → {path}')

print('\n=== Per-stage trajectory summary ===')
summary_rows = []
for name, summary in STAGE_SUMMARIES.items():
    if not summary:
        continue
    summary_rows.append({
        'stage': name,
        'episodes': summary['episodes'],
        'mean_reward': round(summary['mean_reward'], 3),
        'best_reward': round(summary['best_reward'], 3),
        'worst_reward': round(summary['worst_reward'], 3),
        'resolution_rate': round(summary['resolution_rate'], 3),
        'mean_steps': round(summary['mean_steps'], 2),
    })
if summary_rows:
    display(pd.DataFrame(summary_rows))

# Show the last stage's reward CSV + plot inline.
if CURRICULUM_STAGES:
    last_stage = CURRICULUM_STAGES[-1]['name']
    last_dir = output_dir / f'stage_{last_stage}'
    last_csv = last_dir / 'reward_log.csv'
    last_png = last_dir / 'reward_plot.png'
    if last_csv.exists():
        display(_normalize_reward_log(last_csv).tail(10))
    if last_png.exists():
        display(Image(filename=str(last_png)))

if PUSH_TO_HUB:
    if 'HF_TOKEN' not in os.environ:
        raise RuntimeError('HF_TOKEN not set. Add it in Colab secrets or env vars.')
    trainer.push_to_hub(repo_id=HUB_REPO)
    print(f'Model pushed to https://huggingface.co/{HUB_REPO}')
else:
    print('Skipping push. Set PUSH_TO_HUB=True to enable.')

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting GRPO curriculum training...
  Model        : Qwen/Qwen3-0.6B
  KL β         : 0.04
  g (num_gen)  : 8
  Episodes/stg : 4
  Environment  : https://neokazuha-pagezero-agent.hf.space
  Stages       : ['easy', 'medium', 'hard']

[curriculum] STAGE = EASY  (episodes=4, β=0.04, g=8, tasks=['task_1', 'task_2', 'task_3', 'task_4', 'task_5'])
* Run finished. Uploading logs to Trackio (please wait...)
* Created new run: pagezero-grpo-2026-04-25_19-07-10-easy
[env-reset] stage=easy task=task_3 scenario_preview='🚨 ALERT: CRITICAL: Redis memory usage > 95%, OOM errors in a'
[env-reset] stage=easy task=task_2 scenario_preview='🚨 ALERT: WARNING: /api/orders endpoint p99 > 3s — sequential'
[env-reset] stage=easy task=task_1 scenario_preview='🚨 ALERT: CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 9'
[env-reset] stage=easy task=task_5 scenario_preview="🚨 ALERT: CRITICAL: 'too many connections' errors, new reques"
[env-reset] stage=easy task=task_4 scenario_preview='🚨 ALERT: WARNING: Cache m

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


  [reward-audit:total] n=8 mean=+0.050 std=0.053 min=+0.000 max=+0.100
  [reward-audit:diag] n=8 mean=+0.050 std=0.053 min=+0.000 max=+0.100
  [reward-audit:fix] n=8 mean=+0.000 std=0.000 min=+0.000 max=+0.000


W0425 19:12:54.095000 1779 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


Step,Training Loss
1,0.000000
2,0.937686
3,0.934062
4,-0.937698
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


[env-reset] stage=easy task=task_1 scenario_preview='🚨 ALERT: CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 9'
[env-reset] stage=easy task=task_1 scenario_preview='🚨 ALERT: CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 9'
[env-reset] stage=easy task=task_1 scenario_preview='🚨 ALERT: CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 9'
[env-reset] stage=easy task=task_1 scenario_preview='🚨 ALERT: CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 9'
[env-reset] stage=easy task=task_1 scenario_preview='🚨 ALERT: CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 9'
[env-reset] stage=easy task=task_4 scenario_preview='🚨 ALERT: WARNING: Cache miss rate spikes to 100% — Redis may'
[env-reset] stage=easy task=task_5 scenario_preview="🚨 ALERT: CRITICAL: 'too many connections' errors, new reques"
[env-reset] stage=easy task=task_4 scenario_preview='🚨 ALERT: WARNING: Cache miss rate spikes to 100% — Redis may'
  [step  1] DIAG check_alerts           r=+0.100 cum=+0.100
  [step  1] DIAG che

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.20 GiB. GPU 0 has a total capacity of 14.56 GiB of which 369.81 MiB is free. Including non-PyTorch memory, this process has 14.20 GiB memory in use. Of the allocated memory 12.63 GiB is allocated by PyTorch, and 1.43 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import subprocess
import os

log_file_path = '/content/PageZero/server.log'
if os.path.exists(log_file_path):
    try:
        result = subprocess.run(['cat', log_file_path], capture_output=True, text=True, check=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error reading log file: {e}")
        print(f"Stderr: {e.stderr}")
    except FileNotFoundError:
        print(f"Log file not found at {log_file_path}")
else:
    print(f"Log file not found at {log_file_path}")

## 9) Adversarial Self-Play (Red Breaker vs Blue Fixer)

This section adds asymmetric self-play on top of the existing GRPO setup:

- **Red agent (breaker):** picks the most damaging task/scenario for Blue.
- **Blue agent (fixer):** the GRPO policy that diagnoses and resolves incidents.
- **Red reward:** increases with unresolved incidents, downtime, and revenue loss.
- **Blue reward:** unchanged (existing environment reward pipeline).

If `GEMINI_API_KEY` is set, Red task selection is delegated to Gemini. Otherwise a deterministic heuristic fallback is used.

In [ ]:
import json
import random
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
import requests

# -------------------- Adversarial config --------------------
ENABLE_SELF_PLAY = True

# Number of Red→Blue alternation rounds per curriculum stage.
# Tighter alternation (fewer rounds, more stages) improves convergence.
SELF_PLAY_ROUNDS_PER_STAGE = 2

RED_FAULT_BUDGET = 1

# Red reward weights (deterministic component from episode outcomes)
RED_REWARD_ALPHA_UNRESOLVED = 1.2   # +1.2 when Blue fails to resolve
RED_REWARD_ALPHA_DOWNTIME = 0.25    # +0.25 per minute of downtime
RED_REWARD_ALPHA_REVENUE = 0.02     # +0.02 per dollar of revenue loss
RED_REWARD_ALPHA_STEPS = 0.03       # +0.03 per step Blue takes (longer = better for Red)

# Gemini reward for Red: ask Gemini to rate how damaging Red's choice was.
# Blended with deterministic: GEMINI_RED_WEIGHT * gemini + (1-weight) * det
USE_GEMINI_RED_AGENT = bool(os.environ.get('GEMINI_API_KEY', '').strip())
GEMINI_MODEL = os.environ.get('GEMINI_MODEL', 'gemini-2.5-flash')
GEMINI_RED_WEIGHT = 0.4   # how much Gemini reward contributes to Red's total

print(f'[adv-config] USE_GEMINI_RED_AGENT: {USE_GEMINI_RED_AGENT}')
print(f'[adv-config] GEMINI_MODEL: {GEMINI_MODEL}')
print(f'[adv-config] GEMINI_RED_WEIGHT: {GEMINI_RED_WEIGHT}')
print(f'[adv-config] SELF_PLAY_ROUNDS_PER_STAGE: {SELF_PLAY_ROUNDS_PER_STAGE}')


@dataclass
class RedAction:
    stage: str
    task_id: str
    rationale: str
    target_failure_mode: str


def _gemini_post(prompt_text: str, timeout: int = 30) -> Optional[dict]:
    """POST a prompt to Gemini and return parsed JSON or None on failure."""
    api_key = os.environ.get('GEMINI_API_KEY', '').strip()
    if not api_key:
        return None
    url = (
        'https://generativelanguage.googleapis.com/v1beta/models/'
        f'{GEMINI_MODEL}:generateContent?key={api_key}'
    )
    body = {
        'contents': [{'role': 'user', 'parts': [{'text': prompt_text}]}],
        'generationConfig': {'temperature': 0.1, 'responseMimeType': 'application/json'},
    }
    try:
        r = requests.post(url, json=body, timeout=timeout)
        r.raise_for_status()
        text = r.json()['candidates'][0]['content']['parts'][0]['text']
        return json.loads(text)
    except Exception as e:
        print(f'[gemini_post] FAILED: {e}')
        return None


class RedBreakerAgent:
    """Adversarial task selector with Gemini-evaluated reward signal.

    Task selection: Gemini (if available) → heuristic fallback.
    Red reward: deterministic outcome score blended with Gemini quality score.
    Learns online from episode stats to prioritize the hardest tasks.
    """

    def __init__(self):
        self.task_stats: Dict[str, Dict[str, float]] = {}
        self._gemini_select_calls = 0
        self._gemini_select_hits = 0
        self._gemini_reward_calls = 0
        self._gemini_reward_hits = 0

    def debug_stats(self) -> str:
        return (
            f'[red-agent] select calls={self._gemini_select_calls} '
            f'hits={self._gemini_select_hits} | '
            f'reward calls={self._gemini_reward_calls} '
            f'hits={self._gemini_reward_hits}'
        )

    def _score_task(self, task_id: str) -> float:
        """Heuristic task score: higher = more adversarial."""
        s = self.task_stats.get(task_id, {})
        success = float(s.get('blue_success_rate', 0.5))
        avg_downtime = float(s.get('avg_downtime', 0.0))
        avg_red = float(s.get('avg_red_reward', 0.0))
        return (1.0 - success) * 1.8 + avg_downtime * 0.08 + avg_red * 0.4

    def _gemini_choose(self, stage_name: str, task_ids: List[str]) -> Optional[RedAction]:
        if not os.environ.get('GEMINI_API_KEY', '').strip():
            return None

        self._gemini_select_calls += 1
        ranked = sorted(task_ids, key=self._score_task, reverse=True)
        stats_view = {
            t: self.task_stats.get(t, {
                'blue_success_rate': 0.5, 'avg_downtime': 0.0,
                'avg_red_reward': 0.0, 'episodes': 0,
            })
            for t in ranked
        }

        prompt = json.dumps({
            'stage': stage_name,
            'budget': RED_FAULT_BUDGET,
            'task_ids': task_ids,
            'task_stats': stats_view,
            'objective': (
                'You are a Red adversarial agent. Choose the ONE task_id that is maximally '
                'adversarial to a Blue SRE fixer. Consider: low blue success rate, high downtime, '
                'cascading failure potential. Return JSON with keys: task_id, rationale, target_failure_mode.'
            ),
            'constraints': [
                'task_id must be exactly one of the provided task_ids',
                'rationale <= 30 words',
                'target_failure_mode: one of latency, oom, lock, cascade, disk, permission',
            ],
        })

        obj = _gemini_post(prompt)
        if obj is None:
            return None

        chosen = obj.get('task_id', '').strip()
        if chosen not in task_ids:
            print(f'[red-agent/gemini] Returned invalid task_id={chosen!r}, fallback')
            return None

        self._gemini_select_hits += 1
        print(
            f'[red-agent/gemini] Selected task_id={chosen} '
            f'rationale="{obj.get("rationale", "")}" '
            f'failure_mode={obj.get("target_failure_mode", "?")}'
        )
        return RedAction(
            stage=stage_name,
            task_id=chosen,
            rationale=obj.get('rationale', 'gemini adversarial pick'),
            target_failure_mode=obj.get('target_failure_mode', 'unknown'),
        )

    def choose(self, stage_name: str, task_ids: List[str]) -> RedAction:
        if USE_GEMINI_RED_AGENT:
            pick = self._gemini_choose(stage_name, task_ids)
            if pick is not None:
                return pick

        ranked = sorted(task_ids, key=self._score_task, reverse=True)
        chosen = ranked[0] if ranked else random.choice(task_ids)
        print(
            f'[red-agent/heuristic] Selected task_id={chosen} '
            f'(score={self._score_task(chosen):.3f})'
        )
        return RedAction(
            stage=stage_name,
            task_id=chosen,
            rationale='heuristic: maximize unresolved+downtime likelihood',
            target_failure_mode='latency/cascading-failure',
        )

    def gemini_evaluate_red_reward(
        self,
        task_id: str,
        episodes: List[dict],
        det_red_reward: float,
    ) -> float:
        """Ask Gemini to rate how damaging Red's choice was for Blue.

        Blended with deterministic outcome reward: GEMINI_RED_WEIGHT * gemini + rest * det.
        Returns final blended red reward. Falls back to det_red_reward if Gemini unavailable.
        """
        self._gemini_reward_calls += 1
        if not os.environ.get('GEMINI_API_KEY', '').strip() or not episodes:
            print(
                f'[red-agent/reward] det_red={det_red_reward:.3f} '
                f'(Gemini skipped — no key or empty episodes)'
            )
            return det_red_reward

        summary = [{
            'is_resolved': ep.get('is_resolved'),
            'num_steps': ep.get('num_steps'),
            'downtime_minutes': ep.get('downtime_minutes', 0.0),
            'revenue_loss_usd': ep.get('revenue_loss_usd', 0.0),
            'blue_total_reward': ep.get('total_reward', 0.0),
            'tool_sequence': (ep.get('tool_sequence') or [])[:8],
        } for ep in episodes[:4]]

        prompt = json.dumps({
            'task': 'Rate how damaging this Red adversarial task was for the Blue SRE fixer.',
            'task_id': task_id,
            'blue_episode_summaries': summary,
            'scoring_guide': {
                '1.0': 'Blue completely failed, high downtime, no resolution',
                '0.7': 'Blue partially failed or took very long',
                '0.4': 'Blue succeeded but struggled',
                '0.1': 'Blue resolved quickly — Red choice was not effective',
            },
            'required_output': {'gemini_red_score': 'float 0.0-1.0', 'reasoning': 'string <= 30 words'},
        })

        obj = _gemini_post(prompt, timeout=30)
        if obj is None:
            print(
                f'[red-agent/reward] det_red={det_red_reward:.3f} '
                f'(Gemini reward call failed → using det)'
            )
            return det_red_reward

        gemini_score = float(obj.get('gemini_red_score', 0.5))
        # Map [0,1] → reward-space [0, RED_REWARD_ALPHA_UNRESOLVED * 1.5] for comparability
        gemini_reward = gemini_score * RED_REWARD_ALPHA_UNRESOLVED * 1.5
        blended = GEMINI_RED_WEIGHT * gemini_reward + (1.0 - GEMINI_RED_WEIGHT) * det_red_reward
        self._gemini_reward_hits += 1

        print(
            f'[red-agent/reward] task={task_id} '
            f'gemini_score={gemini_score:.3f} gemini_r={gemini_reward:.3f} '
            f'det_r={det_red_reward:.3f} blended={blended:.3f} '
            f'reasoning="{obj.get("reasoning", "")}"'
        )
        return blended

    def update(self, task_id: str, episodes: List[dict], red_reward: Optional[float] = None) -> None:
        if not episodes:
            return
        s = self.task_stats.setdefault(task_id, {
            'episodes': 0, 'blue_success_rate': 0.5,
            'avg_downtime': 0.0, 'avg_red_reward': 0.0,
        })
        n0 = float(s['episodes'])
        n = n0 + len(episodes)
        successes = sum(1 for ep in episodes if bool(ep.get('is_resolved')))
        mean_success = successes / len(episodes)
        mean_downtime = (
            sum(float(ep.get('downtime_minutes', 0.0)) for ep in episodes) / len(episodes)
        )
        mean_red = red_reward if red_reward is not None else (
            sum(float(ep.get('red_reward', 0.0)) for ep in episodes) / len(episodes)
        )
        s['blue_success_rate'] = (s['blue_success_rate'] * n0 + mean_success * len(episodes)) / n
        s['avg_downtime'] = (s['avg_downtime'] * n0 + mean_downtime * len(episodes)) / n
        s['avg_red_reward'] = (s['avg_red_reward'] * n0 + mean_red * len(episodes)) / n
        s['episodes'] = int(n)
        print(
            f'[red-agent/update] task={task_id} '
            f'blue_success={s["blue_success_rate"]:.2f} '
            f'avg_downtime={s["avg_downtime"]:.1f}m '
            f'avg_red_r={s["avg_red_reward"]:.3f} '
            f'total_eps={s["episodes"]}'
        )


class AdversarialPageZeroToolEnv(PageZeroToolEnv):
    """Drop-in env wrapper with Red task override + red reward accounting.

    Sends adversarial_metadata to the server so the backend judge
    (server/llm_judge.py) can calibrate scoring for harder adversarial episodes.
    """

    RED_TASK_OVERRIDE: Optional[str] = None
    RED_METADATA: Dict[str, Any] = {}

    @classmethod
    def set_red_attack(cls, task_id: Optional[str], metadata: Optional[Dict[str, Any]] = None):
        cls.RED_TASK_OVERRIDE = task_id
        cls.RED_METADATA = metadata or {}

    def reset(self, **kwargs) -> str | None:
        if self.RED_TASK_OVERRIDE:
            self.__class__.STAGE_TASK_IDS = [self.RED_TASK_OVERRIDE]
        text = super().reset(**kwargs)
        self.red_metadata = dict(self.RED_METADATA)
        self.red_reward = 0.0
        print(
            f'[adv-env/reset] Red task override={self.RED_TASK_OVERRIDE} '
            f'metadata={self.red_metadata}'
        )
        return text

    def trajectory_payload(self) -> dict:
        base = super().trajectory_payload()
        last_step = self.trajectory[-1] if self.trajectory else {}
        downtime = float(last_step.get('downtime_minutes', 0.0))
        revenue = float(last_step.get('revenue_loss_usd', 0.0))
        unresolved = 0.0 if bool(base.get('is_resolved')) else 1.0
        steps = float(base.get('num_steps', 0))

        det_red_reward = (
            RED_REWARD_ALPHA_UNRESOLVED * unresolved
            + RED_REWARD_ALPHA_DOWNTIME * downtime
            + RED_REWARD_ALPHA_REVENUE * revenue
            + RED_REWARD_ALPHA_STEPS * steps
        )
        self.red_reward = float(det_red_reward)

        print(
            f'[adv-env/trajectory] resolved={bool(base.get("is_resolved"))} '
            f'downtime={downtime:.1f}m revenue=${revenue:.0f} steps={int(steps)} '
            f'det_red_r={det_red_reward:.3f}'
        )

        base['red_reward'] = self.red_reward
        base['det_red_reward'] = det_red_reward
        base['downtime_minutes'] = downtime
        base['revenue_loss_usd'] = revenue
        base['red_action'] = self.red_metadata
        return base


def validate_reward_structure_with_gemini(
    samples: List[dict],
    out_path: Path,
    stage: str = '',
) -> dict:
    """Run per-stage reward-logic sanity audit with Gemini.

    Checks:
    - Does higher blue success correlate with lower red reward?
    - Are reward bucket decompositions consistent with total_reward?
    - Are there suspicious reward spikes?

    Prints a clear PASS/WARN/FAIL verdict with issue list.
    """
    report = {
        'stage': stage,
        'checked_samples': len(samples),
        'gemini_used': False,
        'verdict': 'not_run',
        'issues': [],
        'suggested_adjustments': [],
    }

    # ── Deterministic pre-checks (always run) ──────────────────────────────
    det_issues = []
    for i, s in enumerate(samples):
        mismatch = abs(float(s.get('bucket_total_mismatch', 0.0)))
        if mismatch > 0.5:
            det_issues.append(
                f'Sample {i}: bucket_total_mismatch={mismatch:.3f} > 0.5 '
                f'(task={s.get("red_task_id","?")})'
            )
        blue_r = float(s.get('blue_total_reward', 0.0))
        red_r = float(s.get('red_reward', 0.0))
        resolved = bool(s.get('blue_resolved'))
        if resolved and red_r > RED_REWARD_ALPHA_UNRESOLVED * 0.8:
            det_issues.append(
                f'Sample {i}: Blue resolved but red_reward={red_r:.3f} seems high'
            )
        if abs(blue_r) > REWARD_AUDIT_MAX:
            det_issues.append(f'Sample {i}: blue_total_reward={blue_r:.3f} out of audit bounds')

    report['det_issues'] = det_issues
    if det_issues:
        print(f'[reward-audit/{stage}] Deterministic issues found:')
        for issue in det_issues:
            print(f'  ⚠ {issue}')
    else:
        print(f'[reward-audit/{stage}] Deterministic checks: all PASS')

    # ── Gemini audit ───────────────────────────────────────────────────────
    api_key = os.environ.get('GEMINI_API_KEY', '').strip()
    if not api_key:
        report['verdict'] = 'det_only_no_gemini'
        report['issues'] = det_issues
        out_path.write_text(json.dumps(report, indent=2))
        print(f'[reward-audit/{stage}] Gemini audit skipped (no API key) → det_only')
        return report

    print(f'[reward-audit/{stage}] Calling Gemini for reward structure audit...')

    prompt = json.dumps({
        'task': 'RL reward structure audit for SRE adversarial self-play',
        'stage': stage,
        'criteria': [
            'Does higher blue success rate correlate with lower red reward?',
            'Are total_reward and bucket decomposition (diag+fix+terminal) consistent?',
            'Are there reward spikes inconsistent with steps or outcome?',
            'Does det_red_reward behave sensibly (higher when Blue fails)?',
        ],
        'samples': samples[:8],
        'required_output': {
            'verdict': 'pass|warn|fail',
            'issues': ['list of specific issue strings'],
            'suggested_adjustments': ['list of concrete reward weight suggestions'],
        },
    })

    obj = _gemini_post(prompt, timeout=45)
    if obj:
        report.update(obj)
        report['gemini_used'] = True
        verdict = report.get('verdict', '?')
        issues = report.get('issues', [])
        adjustments = report.get('suggested_adjustments', [])

        verdict_emoji = {'pass': '✅', 'warn': '⚠️', 'fail': '❌'}.get(verdict, '?')
        print(f'[reward-audit/{stage}] Gemini verdict: {verdict_emoji} {verdict.upper()}')
        for issue in issues:
            print(f'  • {issue}')
        for adj in adjustments:
            print(f'  → {adj}')
    else:
        report['verdict'] = 'warn'
        report['issues'] = det_issues + ['gemini_audit_api_failed']
        print(f'[reward-audit/{stage}] Gemini audit FAILED → using det results only')

    out_path.write_text(json.dumps(report, indent=2))
    print(f'[reward-audit/{stage}] Report saved to {out_path}')
    return report


print('=' * 60)
print('Adversarial self-play utilities initialized.')
print(f'  USE_GEMINI_RED_AGENT : {USE_GEMINI_RED_AGENT}')
print(f'  GEMINI_MODEL         : {GEMINI_MODEL}')
print(f'  GEMINI_RED_WEIGHT    : {GEMINI_RED_WEIGHT}')
print(f'  Rounds/stage         : {SELF_PLAY_ROUNDS_PER_STAGE}')
print('=' * 60)

## 10) Run Adversarial Self-Play Training + Reward Validation

This cell alternates Red and Blue each curriculum stage:

1. Red picks an adversarial task (Gemini if available, else heuristic).
2. Blue trains for one round against that task.
3. Episode trajectories are scored for both Blue and Red rewards.
4. Reward structure is validated and audited (including a Gemini audit when available).

In [ ]:
import gc
import pandas as pd
from pathlib import Path
from datasets import Dataset
from trl import GRPOTrainer

if not ENABLE_SELF_PLAY:
    print('ENABLE_SELF_PLAY=False, skipping adversarial run.')
else:
    adv_output_dir = output_dir / 'adversarial_self_play'
    adv_output_dir.mkdir(parents=True, exist_ok=True)

    # Rebuild trainer using AdversarialPageZeroToolEnv.
    # Same model/tokenizer/LoRA config as the normal trainer — only the
    # environment_factory changes so adversarial metadata flows server-side.
    adv_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[reward_total, reward_diagnosis, reward_fix],
        train_dataset=make_stage_dataset(NUM_EPISODES),
        args=grpo_config,
        peft_config=peft_config,
        environment_factory=AdversarialPageZeroToolEnv,
    )

    red_agent = RedBreakerAgent()
    self_play_records = []
    stage_audit_reports = {}

    print(f'[self-play] Starting adversarial training: {len(CURRICULUM_STAGES)} stages')
    print(f'[self-play] Gemini Red reward: {"enabled" if USE_GEMINI_RED_AGENT else "disabled (heuristic only)"}')

    for stage in CURRICULUM_STAGES:
        stage_name = stage['name']
        stage_dir = adv_output_dir / f'stage_{stage_name}'
        stage_dir.mkdir(parents=True, exist_ok=True)

        bind_logs(
            reward_csv=stage_dir / 'reward_log.csv',
            trajectory_jsonl=stage_dir / TRAJECTORY_LOG_NAME,
            stage=f'{stage_name}_self_play',
        )

        stage_records_this = []  # records accumulated for this stage's audit

        for round_idx in range(SELF_PLAY_ROUNDS_PER_STAGE):
            # ── Red chooses adversarial task ──────────────────────────────
            red_action = red_agent.choose(stage_name, stage['task_ids'])
            adv_metadata = {
                'stage': stage_name,
                'task_id': red_action.task_id,
                'rationale': red_action.rationale,
                'target_failure_mode': red_action.target_failure_mode,
                'round': round_idx + 1,
            }
            AdversarialPageZeroToolEnv.set_stage(stage_name, [red_action.task_id])
            AdversarialPageZeroToolEnv.set_red_attack(red_action.task_id, adv_metadata)

            round_dir = stage_dir / f'round_{round_idx + 1}'
            adv_trainer.train_dataset = make_stage_dataset(stage['episodes'])
            adv_trainer.args.output_dir = str(round_dir)
            adv_trainer.args.run_name = (
                f'pagezero-adv-{timestamp}-{stage_name}-r{round_idx + 1}'
            )
            if hasattr(adv_trainer.state, 'global_step'):
                adv_trainer.state.global_step = 0

            print()
            print('=' * 80)
            print(
                f'[self-play] STAGE={stage_name.upper()} '
                f'ROUND={round_idx + 1}/{SELF_PLAY_ROUNDS_PER_STAGE}'
            )
            print(f'  Red task      : {red_action.task_id}')
            print(f'  Red rationale : {red_action.rationale}')
            print(f'  Failure mode  : {red_action.target_failure_mode}')
            print(f'  Gemini Red    : {"enabled" if USE_GEMINI_RED_AGENT else "heuristic"}')
            print('=' * 80)

            # ── Blue trains against Red's chosen task ─────────────────────
            try:
                adv_trainer.train()
            finally:
                AdversarialPageZeroToolEnv._close_all()

            # ── Collect this round's episodes ─────────────────────────────
            traj_file = stage_dir / TRAJECTORY_LOG_NAME
            round_episodes = []
            if traj_file.exists():
                with open(traj_file) as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            row = json.loads(line)
                        except Exception:
                            continue
                        if row.get('stage') == f'{stage_name}_self_play':
                            round_episodes.append(row)
            tail = round_episodes[-stage['episodes']:] if round_episodes else []

            # ── Compute Red reward (det + Gemini blend) ───────────────────
            for ep in tail:
                det_r = float(ep.get('det_red_reward', ep.get('red_reward', 0.0)))
                ep['det_red_reward'] = det_r

            det_red_mean = (
                sum(float(ep.get('det_red_reward', 0.0)) for ep in tail) / max(1, len(tail))
            )
            blended_red = red_agent.gemini_evaluate_red_reward(
                red_action.task_id, tail, det_red_mean
            )
            print(f'[self-play] Round red reward: det={det_red_mean:.3f} blended={blended_red:.3f}')

            # ── Update Red agent stats ────────────────────────────────────
            red_agent.update(red_action.task_id, tail, red_reward=blended_red)
            print(red_agent.debug_stats())

            # ── Accumulate records ────────────────────────────────────────
            for ep in tail:
                record = {
                    'stage': stage_name,
                    'round': round_idx + 1,
                    'red_task_id': red_action.task_id,
                    'red_rationale': red_action.rationale,
                    'red_failure_mode': red_action.target_failure_mode,
                    'blue_total_reward': ep.get('total_reward', 0.0),
                    'blue_diag_reward': ep.get('diagnosis_reward', 0.0),
                    'blue_fix_reward': ep.get('fix_reward', 0.0),
                    'blue_resolved': int(bool(ep.get('is_resolved'))),
                    'blue_steps': ep.get('num_steps', 0),
                    'downtime_minutes': ep.get('downtime_minutes', 0.0),
                    'revenue_loss_usd': ep.get('revenue_loss_usd', 0.0),
                    'det_red_reward': ep.get('det_red_reward', 0.0),
                    'blended_red_reward': blended_red,
                    'bucket_total_mismatch': ep.get('bucket_total_mismatch', 0.0),
                }
                self_play_records.append(record)
                stage_records_this.append(record)

            # ── Save round checkpoint ─────────────────────────────────────
            ckpt = round_dir / 'checkpoint'
            ckpt.mkdir(parents=True, exist_ok=True)
            adv_trainer.save_model(str(ckpt))
            print(f'[self-play] Round checkpoint saved: {ckpt}')

            gc.collect()
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass

        # ── Per-stage reward audit (after all rounds in this stage) ───────
        print(f'\n[self-play] Running per-stage reward audit for stage={stage_name}...')
        audit_path = stage_dir / 'gemini_reward_audit.json'
        audit = validate_reward_structure_with_gemini(
            stage_records_this, audit_path, stage=stage_name
        )
        stage_audit_reports[stage_name] = audit

        # Plot reward curve for this stage.
        try:
            plot_rewards(stage_dir / 'reward_log.csv', stage_dir / 'reward_plot.png')
        except Exception as e:
            print(f'[self-play] Could not plot stage rewards: {e}')

    # ── Final save and summary ─────────────────────────────────────────────
    records_path = adv_output_dir / 'self_play_records.csv'
    pd.DataFrame(self_play_records).to_csv(records_path, index=False)
    print(f'\nSelf-play records saved to: {records_path}')

    # Global reward invariant check.
    if self_play_records:
        mismatches = [abs(float(r.get('bucket_total_mismatch', 0.0))) for r in self_play_records]
        max_mismatch = max(mismatches)
        print(f'[reward-check] max |bucket_total_mismatch| across all stages = {max_mismatch:.4f}')
        if max_mismatch > 1.0:
            print('[reward-check] WARNING: large decomposition mismatch — review llm_judge.py')

    print('\n=== Adversarial Self-Play Stage Audit Summary ===')
    for sname, audit in stage_audit_reports.items():
        verdict = audit.get('verdict', '?')
        gemini = '✅ Gemini' if audit.get('gemini_used') else '⚠ det-only'
        print(f'  {sname:6s} → {verdict.upper():4s} [{gemini}]')

    adv_final_dir = adv_output_dir / 'final'
    adv_final_dir.mkdir(parents=True, exist_ok=True)
    adv_trainer.save_model(str(adv_final_dir))
    print(f'\nAdversarial self-play model saved to: {adv_final_dir}')
    print(f'Red agent final stats: {red_agent.debug_stats()}')

## 8) Latest Run Snapshot (2026-04-25)

**Run config**
- Model: `Qwen/Qwen3-1.7B`
- Episodes: `6`
- Generations: `6`
- Environment: `https://pranayy-pagezero-agent.hf.space/`
- Runtime: `36/36` steps in `46:43` (Epoch `1/1`)

**Training loss trace (steps 1-36)**

```text
1: 0.000000, 2: 0.228079, 3: -0.283213, 4: 0.005372, 5: 0.000000, 6: 0.000000,
7: 0.000000, 8: 0.000000, 9: 0.000000, 10: 0.062890, 11: 0.000000, 12: -0.297732,
13: 0.000000, 14: 0.000002, 15: 0.000001, 16: 0.000000, 17: 0.000000, 18: 0.000003,
19: 0.000000, 20: 0.000000, 21: 0.000004, 22: 0.000000, 23: 0.000000, 24: 0.000000,
25: 0.000000, 26: 0.000000, 27: 0.000000, 28: 0.000000, 29: 0.000000, 30: 0.000000,
31: 0.020289, 32: 0.281001, 33: -0.190548, 34: 0.000000, 35: -0.202171, 36: 0.000000
```

**Saved model**
- `outputs/pagezero-colab-2026-04-25_14-46-12`

**Reward curve summary (from plot)**
- Episodes: `36`
- Final average reward: `0.02`
- Best episode reward: `0.40`
- Trend: slightly positive, but near-flat and high-variance.

**Reward log tail**

| episode | total_reward | diagnosis_reward | fix_reward | timestamp |
|---:|---:|---:|---:|---|
| 27 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537719 |
| 28 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537743 |
| 29 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537767 |
| 30 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537790 |
| 31 | 0.2 | 0.0 | 0.2 | 2026-04-25T15:53:59.121829 |
| 32 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121913 |
| 33 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121943 |
| 34 | -0.2 | 0.0 | -0.2 | 2026-04-25T15:53:59.121967 |
| 35 | 0.2 | 0.2 | 0.0 | 2026-04-25T15:53:59.121991 |
| 36 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.122014 |


Latest run log placeholder

## Latest Run Snapshot (2026-04-25)

- Model: `Qwen/Qwen3-1.7B`
- Episodes: `6`
- Generations: `6`
- Environment: `https://pranayy-pagezero-agent.hf.space/`
- Runtime: `36/36` steps in `46:43`
- Saved model: `outputs/pagezero-colab-2026-04-25_14-46-12`
- Reward summary: episodes `36`, final avg `0.02`, best `0.40`.

## 8) Latest Run Snapshot (2026-04-25)

**Run config**
- Model: `Qwen/Qwen3-1.7B`
- Episodes: `6`
- Generations: `6`
- Environment: `https://pranayy-pagezero-agent.hf.space/`
- Runtime: `36/36` steps in `46:43` (Epoch `1/1`)

**Training loss trace (steps 1-36)**

```text
1: 0.000000, 2: 0.228079, 3: -0.283213, 4: 0.005372, 5: 0.000000, 6: 0.000000,
7: 0.000000, 8: 0.000000, 9: 0.000000, 10: 0.062890, 11: 0.000000, 12: -0.297732,
13: 0.000000, 14: 0.000002, 15: 0.000001, 16: 0.000000, 17: 0.000000, 18: 0.000003,
19: 0.000000, 20: 0.000000, 21: 0.000004, 22: 0.000000, 23: 0.000000, 24: 0.000000,
25: 0.000000, 26: 0.000000, 27: 0.000000, 28: 0.000000, 29: 0.000000, 30: 0.000000,
31: 0.020289, 32: 0.281001, 33: -0.190548, 34: 0.000000, 35: -0.202171, 36: 0.000000
```

**Saved model**
- `outputs/pagezero-colab-2026-04-25_14-46-12`

**Reward curve summary (from plot)**
- Episodes: `36`
- Final average reward: `0.02`
- Best episode reward: `0.40`
- Trend: slightly positive, but near-flat and high-variance.

**Reward log tail**

| episode | total_reward | diagnosis_reward | fix_reward | timestamp |
|---:|---:|---:|---:|---|
| 27 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537719 |
| 28 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537743 |
| 29 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537767 |
| 30 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537790 |
| 31 | 0.2 | 0.0 | 0.2 | 2026-04-25T15:53:59.121829 |
| 32 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121913 |
| 33 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121943 |
| 34 | -0.2 | 0.0 | -0.2 | 2026-04-25T15:53:59.121967 |
| 35 | 0.2 | 0.2 | 0.0 | 2026-04-25T15:53:59.121991 |
| 36 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.122014 |

